# 3. Incident Lifecycle: Assign, Classify, Tune

An investigation isn't finished when you **know** what happened — it's finished when the incident is properly **closed**. SC-200 tests this heavily: you must know the classification taxonomy and how closing an incident feeds learning back into the system.

This notebook walks the lifecycle `New → Active → Closed` with assignment, comments, classification, and rule tuning — plus the bad-vs-best contrast.

> **SC-200 mapping**: "Manage incidents", "Classify and tune analytics rules".


## Setup

This lab reuses the mini-SIEM from Lab 1. Make sure it's running:

```bash
cd ../../01-build-a-siem && docker compose up -d
```

Then, in VS Code:
1. Pick the **`.venv` kernel** from this folder (top-right kernel picker).
2. If it's missing, reload the window (`Cmd+Shift+P` → `Reload Window`).

All cells talk to `http://localhost:8000` — the same mini-SIEM container seeded with a realistic multi-stage attack plus normal background traffic.


In [1]:
import httpx
from collections import Counter, defaultdict
from datetime import datetime

SIEM = 'http://localhost:8000'

# Sanity check: can we reach the SIEM?
health = httpx.get(f'{SIEM}/health').json()
print('SIEM health:', health)

dashboard = httpx.get(f'{SIEM}/dashboard').json()
print('Dashboard:', dashboard)


SIEM health: {'status': 'ok', 'service': 'mini-siem', 'counts': {'logs': 172, 'analytics_rules': 6, 'alerts': 6, 'incidents': 6, 'playbooks': 5}}
Dashboard: {'total_logs': 172, 'tables': ['AzureFirewall', 'DeviceEvents', 'EmailEvents', 'SigninLogs'], 'active_rules': 6, 'open_alerts': 0, 'open_incidents': 0, 'severity_breakdown': {'High': 4, 'Medium': 2}}


## The classification taxonomy

When you close an incident you **must** classify it. This trains the ML and gives engineering a signal to tune rules.

| Classification | When to use | Example |
|---|---|---|
| **True Positive** | Real malicious activity, action required | Confirmed phishing + credential theft |
| **Benign Positive** | Real activity, but not malicious | Authorized pen-test triggered an alert |
| **False Positive** | The detection was wrong | Legitimate admin tool flagged as malware |
| **Undetermined** | Not enough evidence to decide yet | Suspicious but isolated signal |

Closing without classifying is the single most common SOC anti-pattern.


## ❌ Bad: silent close

The bad analyst just flips the status to `Closed`. No comment. No classification. No one learns anything.


In [2]:
incidents = httpx.get(f'{SIEM}/incidents').json()

# We'll demonstrate on a lower-severity incident so we don't close the real attack
target = next((i for i in incidents if i['severity'] != 'High' and i['status'] == 'New'), incidents[-1])
print(f"Demo target: {target['id']}  {target['title']}")

# BAD: just close it
httpx.patch(f"{SIEM}/incidents/{target['id']}", json={'status': 'Closed'})

closed = httpx.get(f"{SIEM}/incidents/{target['id']}").json()
print(f"\n❌ status={closed['status']}  classification={closed['classification']!r}  comments={len(__import__('json').loads(closed['comments']))} entries")


Demo target: INC-1c3741  Incident: Outbound traffic to known malicious IP (multiple)

❌ status=Closed  classification='TruePositive'  comments=1 entries


## ✅ Best: assign → comment → classify → close

A mature close does four things, in this order:

1. **Assign** the incident to a named analyst (accountability).
2. **Comment** with the timeline and the "why" of the classification (institutional memory).
3. **Classify** so the detection engine can learn.
4. **Close** the incident.

Our mini-SIEM exposes all four through `PATCH /incidents/{id}`. That endpoint is **not idempotent** — every call appends to comments and overwrites the other fields — so below we read the incident first and only update the fields that need updating.


In [3]:
def mature_close(incident_id: str, analyst: str, summary: str, classification: str):
    current = httpx.get(f'{SIEM}/incidents/{incident_id}').json()

    # 1. Assign (only if not yet assigned — keeps reruns clean)
    if not current.get('assigned_to'):
        httpx.patch(f'{SIEM}/incidents/{incident_id}', json={'assigned_to': analyst})

    # 2. Add the summary comment (only once, to avoid duplicates on rerun)
    import json as _json
    existing = _json.loads(current.get('comments') or '[]')
    if not any(summary == c.get('text') for c in existing):
        httpx.patch(f'{SIEM}/incidents/{incident_id}', json={'comment': summary})

    # 3. Classify (only if not already classified)
    if not current.get('classification'):
        httpx.patch(f'{SIEM}/incidents/{incident_id}', json={'classification': classification})

    # 4. Close
    httpx.patch(f'{SIEM}/incidents/{incident_id}', json={'status': 'Closed'})

    return httpx.get(f'{SIEM}/incidents/{incident_id}').json()

# Walk every "New" incident and close it maturely
for inc in httpx.get(f'{SIEM}/incidents').json():
    if inc['status'] != 'New':
        continue
    cls = 'TruePositive' if inc['severity'] in ('High', 'Critical') else 'BenignPositive'
    result = mature_close(
        inc['id'],
        analyst='analyst1@contoso.com',
        summary=f"Triaged {inc['title']}. Evidence correlated across sources. Classified as {cls}.",
        classification=cls,
    )
    print(f"✅ {result['id']:<12} status={result['status']:<7} class={result['classification']:<14} assigned={result['assigned_to']}")


## Tuning: the forgotten step

If you classify an incident as **False Positive**, the job isn't done — you should also **tune the rule** so it stops firing on the same false pattern. Otherwise you'll keep closing the same FP over and over.

Common tuning moves in Sentinel/Defender XDR:

| Tuning move | When to use |
|---|---|
| **Raise the threshold** | Rule fires on benign bursts of activity |
| **Add an exclusion filter** | A specific admin tool or scanner is triggering it |
| **Suppress for an entity** | A specific service account generates expected traffic |
| **Lower the severity** | The pattern is suspicious but rarely a real threat |
| **Disable the rule** | The rule is obsolete (only as a last resort) |

Let's simulate tuning: show that a `False Positive` incident points us at a rule we should adjust.


In [4]:
fps = [i for i in httpx.get(f'{SIEM}/incidents').json() if i.get('classification') == 'FalsePositive']
print(f'False-positive incidents: {len(fps)}')
if fps:
    for i in fps:
        print(f"  → tune rule behind: {i['title']}")
else:
    print('None in this seed data — but in a real SOC, every FP should trigger a rule review within the week.')

# Show the rules that would be candidates for tuning review
rules = httpx.get(f'{SIEM}/rules').json()
print('\nActive rules (candidates for tuning if misfiring):')
for r in rules:
    print(f"  {r['name']:<34}  sev={r['severity']:<6} threshold={r['threshold']:<3} window={r['window_minutes']}m")


False-positive incidents: 0
None in this seed data — but in a real SOC, every FP should trigger a rule review within the week.

Active rules (candidates for tuning if misfiring):
  Brute force sign-in                 sev=High   threshold=5   window=60m
  Sign-in from suspicious location    sev=Medium threshold=1   window=60m
  Suspicious process execution        sev=High   threshold=1   window=120m
  Lateral movement detected           sev=High   threshold=2   window=60m
  Phishing email delivered            sev=Medium threshold=1   window=120m
  Outbound traffic to known malicious IP  sev=High   threshold=3   window=60m


## What you just did (SC-200 mapping)

| You did... | Real portal equivalent |
|---|---|
| `PATCH /incidents/{id}` with `status/assigned_to/classification/comment` | Incident page → Manage incident panel |
| Guarded against duplicate comments on rerun | Real portals don't dedupe for you either — discipline matters |
| Listed rule parameters for tuning | Analytics rule → Edit → Query logic / thresholds |

### Exam tips

- Know the **four classifications** by name and when to use each.
- "Close" is **not** a classification — it's a status. Always classify.
- **False Positive ⇒ tune the rule.** Closing FPs without tuning is how alert fatigue starts.
- In Defender XDR, closing an incident closes all underlying alerts automatically.

➡️ Next: [04 — Automated response & IOCs](04_automated_response.ipynb)
